# Human Action Recognition: the Final Model (CLIP ViT-B/16, variant J)

This notebook trains the **final model** and studies it: learning curves, test results, **explainable AI**
(what the model looks at), predictions on photos without labels, and **5-fold cross-validation** of the same recipe.

## Why variant J
Notebook 07 trained ten versions of CLIP ViT-B/16, each changing one setting. Results at each one's kept epoch:

| Variant | What it changed | Gap (train − val) | Validation | Test |
|---|---|---|---|---|
| G | strong augmentation | 4.1 | **92.0%** | 92.0% |
| A | baseline (6 blocks frozen) | 2.9 | 91.8% | 91.6% |
| E | dropout 0.3 | 3.0 | 91.6% | 91.9% |
| D | learning rate 2e-5 | 3.5 | 91.5% | 92.0% |
| H | mixup / cutmix | 3.5 | 91.5% | 92.3% |
| I | stop on validation loss | 4.4 | 91.3% | 92.6% |
| **J** | **combined** | **1.9** | **91.2%** | **91.1%** |
| F | drop path 0.3 | 2.8 | 91.0% | 90.9% |
| B | 10 blocks frozen | 2.0 | 90.9% | 90.7% |
| C | all 12 frozen (classifier only) | 1.3 | 88.1% | 89.4% |

- All ten had a gap of 5 points or less (a good fit), so notebook 07's rule ended up choosing by validation accuracy alone (G).
  But the top eight are within 1 point of each other, and **1 point is about 13 of the 1,257 validation photos**:
  within the random variation of a single run.
- So the tie-break is: **among the variants within 1 point of the best validation accuracy, take the smallest gap → J**
  (gap 1.9 points, validation 91.2%). Its curves rose steadily for 11 epochs before flattening.
- **C** had the smallest gap overall, but its training accuracy (89.5%) is lower than the *validation* accuracy of every other
  variant. With only the classifier trainable, the model can't learn more: it **underfits**.
- The test set was not used to choose.

Retraining gives numbers within a few tenths of a point of notebook 07's, because GPU arithmetic is not exactly repeatable.

> **On Kaggle:** **Accelerator → GPU T4 x2**, **Internet → On**, attach the *Human Action Recognition (HAR) Dataset*,
> then **Save Version → Save & Run All**. Six trainings (the final model + 5 folds) run two at a time, one per GPU:
> about **1 hour** (about 2 hours with a single GPU). Finished trainings are saved; if the session stops, running again skips them.

# Part 0 — Settings
Variant J's settings. Each one was first tested on its own in notebook 07 (the variant letter in the last column).

| Setting | Value | What it does | Tested alone |
|---|---|---|---|
| `FREEZE_BLOCKS` | 8 | the lowest 8 of the 12 transformer blocks keep their pretrained weights; the top 4 blocks and the classifier learn | B |
| `LEARNING_RATE` | 3e-5 | the step size at the top of the network; each block below gets ×0.75, the new classifier ×5 | D |
| `DROPOUT` | 0.2 | switches off 20% of the features before the classifier during training | E |
| `DROP_PATH` | 0.2 | randomly skips whole transformer blocks during training | F |
| `WEIGHT_DECAY` | 0.05 | AdamW shrinks the weights slightly every step | all |
| Strong augmentation | on | bigger random changes to each training photo (Part 3.1) | G |
| Mixup / CutMix | on | blends pairs of photos and their labels (Part 3.1) | H |
| `EPOCHS`, `PATIENCE` | 25, 3 | at most 25 epochs; stop after 3 epochs without a better validation accuracy | all |

In [ ]:
EPOCHS = 25            # maximum epochs; early stopping usually ends sooner
PATIENCE = 3           # stop after 3 epochs without a better validation accuracy
FREEZE_BLOCKS = 8      # the lowest 8 of the 12 transformer blocks stay frozen
LEARNING_RATE = 3e-5   # top of the network; each block below x0.75, the classifier x5
DROPOUT = 0.2          # dropout before the classifier
DROP_PATH = 0.2        # drop path (stochastic depth)
WEIGHT_DECAY = 0.05    # AdamW weight decay
LABEL_SMOOTHING = 0.1  # softens the targets
EMA_DECAY = 0.998      # weight averaging: keeps the curves smooth
CV_FOLDS = 5           # cross-validation folds (Part 9)

# Part 1 — Data

## 1.1 Import libraries
This notebook uses **PyTorch** (a deep-learning framework, like Keras/TensorFlow) and **timm**
("PyTorch Image Models"), a library with hundreds of pretrained image networks, including CLIP.

In [ ]:
import os, gc, glob, copy, math, time, random, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", message="Detected call of")   # harmless scheduler notice with mixed precision

SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Device: ", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU - turn on the GPU accelerator in Settings!")

## 1.2 Find the dataset

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
OUTPUT_DIR = "/kaggle/working"
print("Data folder:", DATA_DIR)

## 1.3 Load the labels
Each action gets a number (`calling → 0`, `clapping → 1`, …), because networks work with numbers.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)
NUM_CLASSES = len(class_names)

print(f"{len(df)} labeled images, {NUM_CLASSES} actions")
print(df["label"].value_counts().sort_index().to_string())

## 1.4 Example images

In [ ]:
sample = df.sample(8, random_state=1)
plt.figure(figsize=(14, 6))
for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"])))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1.5 Find near-duplicate photos (prevent data leakage)
Some photos in the dataset are near-copies of each other (the same picture saved twice, resized or recompressed).
If one copy lands in the **training** set and the other in the **test** set, the model can simply *remember*
the photo, and the test accuracy looks better than it really is. This is called **data leakage**.

To prevent it we give every photo a **fingerprint** with *difference hashing (dHash)*:
1. Shrink the photo to 9 × 8 pixels in grayscale.
2. For each pixel, record whether it is brighter than its right-hand neighbour → **64 yes/no bits**.

Photos whose fingerprints differ in **4 bits or fewer** are put in the same **group**, and a group is never
split across train, validation and test.

The fingerprint is deliberately **cautious**. Besides real copies, it also matches some *different* photos that
share a large plain background (white studio shots, black borders), because plain areas give identical bits.
Keeping two different photos on the same side of the split doesn't leak anything, so that's harmless.
To see how many matches are real copies, we also compare small 32 × 32 colour thumbnails pixel by pixel.

In [ ]:
def dhash(path):
    img = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    px = np.asarray(img, dtype=np.int16)
    return (px[:, 1:] > px[:, :-1]).flatten()          # 8 rows x 8 comparisons = 64 bits

paths = [os.path.join(TRAIN_IMG_DIR, f) for f in df["filename"]]
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    bits = np.array(list(pool.map(dhash, paths)), dtype=np.float32)   # shape: (images, 64)

# Hamming distance = number of bits that differ between two fingerprints
pair_a, pair_b = [], []
for start in range(0, len(bits), 1024):
    chunk = bits[start:start + 1024]
    agree = chunk @ bits.T + (1 - chunk) @ (1 - bits).T
    a, b = np.nonzero(64 - agree <= 4)
    a = a + start
    keep = b > a                                        # count each pair once, skip self-matches
    pair_a.append(a[keep])
    pair_b.append(b[keep])
pair_a, pair_b = np.concatenate(pair_a), np.concatenate(pair_b)

n = len(df)
_, groups = connected_components(coo_matrix((np.ones(len(pair_a)), (pair_a, pair_b)), shape=(n, n)), directed=False)
df["group"] = groups

# How alike are the matched pairs really? Average pixel difference of 32 x 32 colour thumbnails (0-255 scale)
def thumbnail(path):
    return np.asarray(Image.open(path).convert("RGB").resize((32, 32), Image.BILINEAR), dtype=np.float32)
pixel_diff = np.array([np.abs(thumbnail(paths[a]) - thumbnail(paths[b])).mean() for a, b in zip(pair_a, pair_b)])
is_copy = pixel_diff < 10
same_label = df["label"].values[pair_a] == df["label"].values[pair_b]

print(f"Photo pairs with matching fingerprints: {len(pair_a)}")
if len(pair_a):
    print(f"  clearly copies (thumbnails differ by less than 10 out of 255): {is_copy.sum()}"
          f" - same label in {same_label[is_copy].mean() if is_copy.any() else 0:.0%} of them")
    print(f"  other matches (near-copies such as crops, and look-alikes):      {(~is_copy).sum()}")
print(f"Groups: {groups.max() + 1} groups for {n} images")

**Examples.** The first three pairs are copies of the same photo; the differences are watermarks or image quality.
The last pair is a *look-alike*: two different photos that the fingerprint matched because of their plain backgrounds.

In [ ]:
copies = np.where(is_copy)[0]
copies = copies[np.argsort(-pixel_diff[copies])][:3]      # copies with small visible differences
lookalikes = np.where(~is_copy)[0]
lookalikes = lookalikes[np.argsort(pixel_diff[lookalikes])][len(lookalikes) // 2:][:1]   # a typical look-alike
rows = [(k, "copy") for k in copies] + [(k, "look-alike") for k in lookalikes]
if rows:
    plt.figure(figsize=(8, 3.2 * len(rows)))
    for r, (k, kind) in enumerate(rows):
        for j, idx in enumerate((pair_a[k], pair_b[k])):
            plt.subplot(len(rows), 2, 2 * r + j + 1)
            plt.imshow(Image.open(paths[idx]))
            title = f"{df['filename'][idx]} · {df['label'][idx]}"
            if j == 0:
                title = f"{kind.upper()} (pixel difference {pixel_diff[k]:.1f})\n" + title
            plt.title(title, fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1.6 Split into training, validation and test sets
- **Training (80%)** – the model learns from these photos.
- **Validation (10%)** – checked after every epoch to pick the best version of the model.
- **Test (10%)** – used **only once at the very end** to report the final accuracy.

Because the validation set is used to *choose* the best epoch, its score is slightly optimistic.
The untouched **test set** gives an honest final number. *Stratified* keeps the same mix of actions in every set;
*group* keeps near-duplicates together.

(The dataset's own `test` folder has no labels, so it can't measure accuracy. Part 6 explains what we use it for.)

In [ ]:
y_all = df["label_id"].values
all_idx = np.arange(len(df))

split_test = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(split_test.split(all_idx, y_all, groups))

split_val = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED)
tr, va = next(split_val.split(trainval_idx, y_all[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group) or set(val_df.group) & set(test_df.group))
print(f"Training:   {len(train_df):>5} images")
print(f"Validation: {len(val_df):>5} images")
print(f"Test:       {len(test_df):>5} images")
print("No near-duplicate group is shared between the sets.")

# Part 2 — Preprocessing

## 2.1 Resize, normalize and augment
**Every image (validation and test):**
- **Resize** to 224 × 224 pixels — the input size CLIP ViT-B/16 was trained on.
- **Normalize** each colour channel with CLIP's own mean and standard deviation, so the inputs look like
  what the network saw during pretraining.

**Training images only — data augmentation** (a new random version every epoch):
- **Random resized crop** — zoom into 40–100% of the photo, so the model sees people at different sizes and positions.
- **Horizontal flip** — a mirrored person is doing the same action.
- **Small rotation** (±10°) and **brightness / contrast** changes (±20%).

Augmentation makes the training set look bigger and more varied, which reduces **overfitting**.

In [ ]:
IMG_SIZE = 224
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

def to_display(t):
    # Undo the normalization so a tensor can be shown as a picture
    mean, std = torch.tensor(CLIP_MEAN).view(3, 1, 1), torch.tensor(CLIP_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

example = Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][0])).convert("RGB")
plt.figure(figsize=(14, 3.5))
plt.subplot(1, 6, 1); plt.imshow(to_display(eval_tf(example))); plt.title("original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2); plt.imshow(to_display(train_tf(example))); plt.title(f"augmented {i + 1}"); plt.axis("off")
plt.suptitle(f"Action: {train_df['label'][0]}")
plt.tight_layout()
plt.show()

## 2.2 Data loaders
A **Dataset** says how to load one image and its label; a **DataLoader** groups images into **batches of 32**
and loads them in parallel on the CPU while the GPU trains.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 2

class HARDataset(Dataset):
    def __init__(self, frame, img_dir, transform, labeled=True):
        self.files = frame["filename"].tolist()
        self.labels = frame["label_id"].tolist() if labeled else [-1] * len(frame)
        self.img_dir, self.transform = img_dir, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.files[i])).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, img_dir, transform, train, labeled=True):
    return DataLoader(HARDataset(frame, img_dir, transform, labeled),
                      batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(train_df, TRAIN_IMG_DIR, train_tf, train=True)
val_loader = make_loader(val_df, TRAIN_IMG_DIR, eval_tf, train=False)
test_loader = make_loader(test_df, TRAIN_IMG_DIR, eval_tf, train=False)

x, y = next(iter(train_loader))
print("Batch of images:", tuple(x.shape), "-> 32 images, 3 colours, 224 x 224 pixels")
print("Batch of labels:", tuple(y.shape))
print("Training steps per epoch:", len(train_loader))

# Part 3 — The Training Recipe
## 3.1 Strong augmentation and mixup / cutmix
Training photos get **strong augmentation**: a random crop (40–100% of the photo), a random flip, one random change from
TrivialAugment (for example brightness, contrast, rotation, sharpness or posterize, at a random strength), and in 25% of
photos a random rectangle is blanked out (random erasing). Validation and test photos are only resized, as in Part 2.

On half of the training batches, **mixup** or **cutmix** combines pairs of photos:
- **Mixup** blends two photos; the label is blended in the same proportion (for example 70% eating + 30% texting).
- **CutMix** pastes a rectangle from one photo into another; the label is shared by area.

The model can't memorise single photos this way, which reduces overfitting. The pictures below show one example of each.

In [ ]:
strong_tf = transforms.Compose([      # the same transform is used in the training script (Part 3.3)
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
    transforms.RandomErasing(p=0.25, value="random"),
])

# Five strongly augmented versions of one training photo
example = Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][0])).convert("RGB")
plt.figure(figsize=(14, 3.5))
plt.subplot(1, 6, 1); plt.imshow(to_display(eval_tf(example))); plt.title("original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2); plt.imshow(to_display(strong_tf(example))); plt.title(f"strong {i + 1}"); plt.axis("off")
plt.suptitle(f"Strong augmentation · action: {train_df['label'][0]}")
plt.tight_layout(); plt.show()

# Mixup and cutmix on two photos with different actions
i, j = 0, int(np.argmax(train_df["label"].values != train_df["label"][0]))
a = eval_tf(Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][i])).convert("RGB"))
b = eval_tf(Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][j])).convert("RGB"))
lam = 0.7
mixed = lam * a + (1 - lam) * b                     # mixup: 70% of photo A + 30% of photo B
cut = a.clone()
cut[:, 62:162, 62:162] = b[:, 62:162, 62:162]       # cutmix: a 100 x 100 square from photo B
share_b = 100 * 100 / IMG_SIZE ** 2
la, lb = train_df["label"][i], train_df["label"][j]
plt.figure(figsize=(14, 4))
for k, (img, title) in enumerate([(a, f"photo A: {la}"), (b, f"photo B: {lb}"),
                                  (mixed, f"mixup label:\n{lam:.0%} {la} + {1 - lam:.0%} {lb}"),
                                  (cut, f"cutmix label:\n{1 - share_b:.0%} {la} + {share_b:.0%} {lb}")]):
    plt.subplot(1, 4, k + 1); plt.imshow(to_display(img)); plt.title(title, fontsize=10); plt.axis("off")
plt.tight_layout(); plt.show()

# In training (Part 3.3): mixup or cutmix on half of the batches, with a random share each time

## 3.2 The six trainings: the final model and the cross-validation folds
1. **The final model** is trained on the split from Part 1.6 (training / validation / test), exactly like variant J in notebook 07.
2. **Cross-validation (Part 9):** all 12,600 labeled photos are split into **5 folds** (stratified, near-duplicate groups kept
   together), the same folds as notebook 03. Each fold is the test set once. The other four folds are split again into
   training (7/8) and validation (1/8), so early stopping has its own validation set and the test fold is never used to choose anything.

The photo lists of each training are saved to files, because the training runs in separate processes (one per GPU).

In [ ]:
import json, subprocess, sys
MODEL_NAME = "vit_base_patch16_clip_224.openai"
PRETRAINED = True
WORK = os.path.join(OUTPUT_DIR, "final")

def save_job(name, train_frame, val_frame, test_frame, seed, save_model):
    folder = os.path.join(WORK, "jobs", name)
    os.makedirs(folder, exist_ok=True)
    train_eval = train_frame.sample(min(2000, len(train_frame)), random_state=SEED)   # re-measured on clean photos
    for part, frame in (("train", train_frame), ("val", val_frame), ("test", test_frame), ("train_eval", train_eval)):
        frame[["filename", "label_id"]].to_csv(os.path.join(folder, f"{part}.csv"), index=False)
    print(f"{name:>7}: {len(train_frame):,} training / {len(val_frame):,} validation / {len(test_frame):,} test photos")
    return {"name": name, "seed": seed, "save_model": save_model}

# Training 1: the final model, on the split from Part 1.6
jobs = [save_job("final", train_df, val_df, test_df, SEED, save_model=True)]

# Trainings 2-6: one per cross-validation fold
folds = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
inner = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=SEED)
for fold, (rest_idx, test_idx_f) in enumerate(folds.split(all_idx, y_all, groups), start=1):
    tr, va = next(inner.split(rest_idx, y_all[rest_idx], groups[rest_idx]))
    fold_train, fold_val, fold_test = (df.iloc[ix].reset_index(drop=True) for ix in (rest_idx[tr], rest_idx[va], test_idx_f))
    assert not (set(fold_train.group) & set(fold_val.group) or set(fold_train.group) & set(fold_test.group)
                or set(fold_val.group) & set(fold_test.group))       # no near-duplicate shared between the sets
    jobs.append(save_job(f"fold_{fold}", fold_train, fold_val, fold_test, SEED + fold, save_model=False))

n_gpu = torch.cuda.device_count()
n_procs = 2 if n_gpu >= 2 else 1
settings = dict(epochs=EPOCHS, patience=PATIENCE, freeze_blocks=FREEZE_BLOCKS, lr=LEARNING_RATE, dropout=DROPOUT,
                drop_path=DROP_PATH, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING, ema_decay=EMA_DECAY)
config = dict(img_dir=TRAIN_IMG_DIR, num_classes=NUM_CLASSES, model_name=MODEL_NAME, pretrained=PRETRAINED,
              img_size=IMG_SIZE, batch_size=BATCH_SIZE, mean=list(CLIP_MEAN), std=list(CLIP_STD),
              workers=max(1, (os.cpu_count() or 2) // n_procs), settings=settings, jobs=jobs)
with open(os.path.join(WORK, "config.json"), "w") as f:
    json.dump(config, f, indent=1)
print(f"{len(jobs)} trainings; GPUs: {n_gpu} -> {n_procs} training process(es) in parallel")

## 3.3 The training script
Every training runs this code. It:
1. Loads CLIP ViT-B/16's image encoder (85.8M weights) with a new classifier: dropout → dense layer with 15 neurons → softmax.
2. Freezes the patch embedding and the lowest `FREEZE_BLOCKS` transformer blocks. The trainable layers get **layer-wise
   learning rates**: `LEARNING_RATE` at the top of the network, ×0.75 for each block below, and ×5 for the new classifier,
   because it starts from random weights while the blocks are already pretrained.
3. Trains with **AdamW** (weight decay `WEIGHT_DECAY`, not on biases and normalization weights), 1 warm-up epoch then
   **cosine decay**, **cross-entropy with soft targets** (the mixed labels from mixup / cutmix, smoothed by 0.1),
   **gradient clipping** at 1.0 and **mixed precision** (FP16).
4. Keeps an **EMA** copy (weights averaged over roughly the last 500 steps), which makes the curves smooth. After every epoch
   it measures the EMA copy the same way on 2,000 clean training photos, the validation set and the test set.
5. **Early stopping:** stops after `PATIENCE` epochs without a better validation accuracy, keeps the best epoch, and saves
   the history, the result and the test predictions (and, for the final model, its weights).

In [ ]:
%%writefile /kaggle/working/train_final.py
import argparse, copy, gc, json, math, os, random, time, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

parser = argparse.ArgumentParser()
parser.add_argument("--work", required=True)           # the folder with the trainings and the settings (Part 3.2)
args = parser.parse_args()
with open(os.path.join(args.work, "config.json")) as f:
    C = json.load(f)
S = C["settings"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
LAYER_DECAY, HEAD_LR_MULT, GRAD_CLIP, WARMUP_EPOCHS = 0.75, 5.0, 1.0, 1

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class Photos(Dataset):
    def __init__(self, frame, tf):
        self.files, self.labels, self.tf = frame["filename"].tolist(), frame["label_id"].tolist(), tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(os.path.join(C["img_dir"], self.files[i])).convert("RGB")), self.labels[i]

size = C["img_size"]
strong_tf = transforms.Compose([                         # strong augmentation (Part 3.1)
    transforms.RandomResizedCrop(size, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(), transforms.Normalize(C["mean"], C["std"]),
    transforms.RandomErasing(p=0.25, value="random")])
eval_tf = transforms.Compose([                           # validation and test photos: only resized
    transforms.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(), transforms.Normalize(C["mean"], C["std"])])

def loader(folder, part, tf, train):
    frame = pd.read_csv(os.path.join(folder, f"{part}.csv"))
    return DataLoader(Photos(frame, tf), batch_size=C["batch_size"] if train else 128, shuffle=train, drop_last=train,
                      num_workers=C["workers"], pin_memory=use_amp, persistent_workers=True)

def layer_id(name, num_blocks):
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "norm_pre")):
        return 0                                  # input embeddings: the lowest layer
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1        # transformer blocks 1-12
    return num_blocks + 1                         # final norm and classifier: the top

def param_groups(net, base_lr, freeze_blocks, weight_decay):
    num_blocks = len(net.blocks)
    head_ids = {id(p) for p in net.get_classifier().parameters()}
    groups = {}
    for name, p in net.named_parameters():
        lid = layer_id(name, num_blocks)
        is_head = id(p) in head_ids
        if not is_head and lid <= freeze_blocks:
            p.requires_grad_(False)                   # frozen: not trained
            continue
        lr = base_lr * HEAD_LR_MULT if is_head else base_lr * LAYER_DECAY ** (num_blocks + 1 - lid)
        no_decay = p.ndim <= 1 or name in ("cls_token", "pos_embed")   # no weight decay on biases and norms
        g = groups.setdefault((lid, is_head, no_decay),
                              {"params": [], "lr": lr, "weight_decay": 0.0 if no_decay else weight_decay})
        g["params"].append(p)
    return list(groups.values())

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))          # averages less at the start
        for ema_v, v in zip(self.model.state_dict().values(), model.state_dict().values()):
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                ema_v.copy_(v)

@torch.no_grad()
def predict(net, dl):
    net.eval()
    probs, labels = [], []
    for x, y in dl:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(net(x.to(device, non_blocking=True)).float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def measure(net, dl):
    p, y = predict(net, dl)
    return F.nll_loss(torch.log(p + 1e-9), y).item(), (p.argmax(1) == y).float().mean().item()

def claim(folder):
    # Writes a "claimed" marker so the other GPU skips this training. O_EXCL: only one process can create it.
    try:
        os.close(os.open(os.path.join(folder, "claimed"), os.O_CREAT | os.O_EXCL))
        return True
    except FileExistsError:
        return False

def train_job(job, folder):
    name = job["name"]
    seed_everything(job["seed"])
    t_start = time.time()
    net = timm.create_model(C["model_name"], pretrained=C["pretrained"], num_classes=C["num_classes"],
                            drop_rate=S["dropout"], drop_path_rate=S["drop_path"]).to(device)
    train_dl = loader(folder, "train", strong_tf, True)
    evals = {"train": loader(folder, "train_eval", eval_tf, False), "val": loader(folder, "val", eval_tf, False),
             "test": loader(folder, "test", eval_tf, False)}
    groups = param_groups(net, S["lr"], S["freeze_blocks"], S["weight_decay"])
    trainable_params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    steps = len(train_dl)
    total, warm = S["epochs"] * steps, WARMUP_EPOCHS * steps
    def lr_factor(t):
        if t < warm:
            return t / warm                                                                  # warm-up
        return max(0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))), 0.01)  # cosine decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    mix = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5,   # on half of the batches
                label_smoothing=S["label_smoothing"], num_classes=C["num_classes"])
    criterion = SoftTargetCrossEntropy()
    ema = EMA(net, S["ema_decay"])
    print(f"[{name}] training {sum(p.numel() for p in trainable_params) / 1e6:.1f}M of "
          f"{sum(p.numel() for p in net.parameters()) / 1e6:.1f}M weights", flush=True)
    history, best_acc, best_epoch, best_state, bad, step = [], -1.0, 0, None, 0, 0
    for epoch in range(1, S["epochs"] + 1):
        t0 = time.time()
        net.train()
        run_loss, run_correct, seen = 0.0, 0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            x, target = mix(x, y)                           # target = soft labels
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = net(x)
                loss = criterion(logits.float(), target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(net, step)
            step += 1
            run_loss += loss.item() * len(y)
            run_correct += (logits.argmax(1) == target.argmax(1)).sum().item()   # the label with the bigger share
            seen += len(y)
        row = {"epoch": epoch, "running_train_loss": run_loss / seen, "running_train_acc": run_correct / seen}
        for part, d in evals.items():
            row[f"{part}_loss"], row[f"{part}_acc"] = measure(ema.model, d)
        history.append(row)
        improved = row["val_acc"] > best_acc
        if improved:
            best_acc, best_epoch, bad = row["val_acc"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
        else:
            bad += 1
        print(f"[{name}] epoch {epoch:2d} | train {row['train_acc']:.1%} (running {row['running_train_acc']:.1%}) | "
              f"val {row['val_acc']:.1%} | test {row['test_acc']:.1%} | val loss {row['val_loss']:.3f} | "
              f"{time.time() - t0:.0f}s{'  <- best' if improved else ''}", flush=True)
        if bad >= S["patience"]:
            print(f"[{name}] early stopping after epoch {epoch}", flush=True)
            break
    ema.model.load_state_dict(best_state)
    h = pd.DataFrame(history)
    b = h[h["epoch"] == best_epoch].iloc[0]
    test_probs, test_labels = predict(ema.model, evals["test"])
    result = {"training": name, "train photos": len(train_dl.dataset), "val photos": len(evals["val"].dataset),
              "test photos": len(evals["test"].dataset), "epochs run": len(h), "best epoch": int(best_epoch),
              "train acc": float(b["train_acc"]), "val acc": float(b["val_acc"]), "test acc": float(b["test_acc"]),
              "train-val gap": float(b["train_acc"] - b["val_acc"]), "minutes": round((time.time() - t_start) / 60, 1)}
    h.to_csv(os.path.join(folder, "history.csv"), index=False)
    np.save(os.path.join(folder, "test_probs.npy"), test_probs.numpy())
    np.save(os.path.join(folder, "test_labels.npy"), test_labels.numpy())
    if job["save_model"]:
        torch.save(ema.model.state_dict(), os.path.join(folder, "model.pt"))
    with open(os.path.join(folder, "result.json"), "w") as f:      # written last: marks the training as finished
        json.dump(result, f)
    print(f"[{name}] DONE: best epoch {best_epoch}, val {result['val acc']:.1%}, test {result['test acc']:.1%}, "
          f"gap {result['train-val gap'] * 100:+.1f} pts, {result['minutes']} min", flush=True)

# Work queue: take the next training that nobody has claimed yet, so both GPUs stay busy until the end
for job in C["jobs"]:
    folder = os.path.join(args.work, "jobs", job["name"])
    if os.path.exists(os.path.join(folder, "result.json")) or not claim(folder):
        continue                                   # finished earlier, or the other GPU has it
    try:
        train_job(job, folder)
    except Exception:
        print(f"[{job['name']}] FAILED", flush=True)
        traceback.print_exc()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Part 4 — Training
One training process per GPU. Each process takes the next training that nobody has started (the final model first),
so both GPUs stay busy. Progress is printed every minute; full logs are saved in `/kaggle/working/final/log_*.txt`.
`running` is the training accuracy *during* the epoch, on augmented and mixed photos: those are much harder, so it is much
lower. The curves in Part 5 use the clean measurement.

In [ ]:
SCRIPT = "/kaggle/working/train_final.py"
POLL_SECONDS = 60

# Trainings started but not finished last time (the session stopped) are trained again
for job in jobs:
    folder = os.path.join(WORK, "jobs", job["name"])
    if os.path.exists(os.path.join(folder, "result.json")):
        print(f"{job['name']}: finished earlier, skipped")
    elif os.path.exists(os.path.join(folder, "claimed")):
        os.remove(os.path.join(folder, "claimed"))

procs = []
for g in range(n_procs):
    env = dict(os.environ)
    if n_gpu:
        env["CUDA_VISIBLE_DEVICES"] = str(g)       # this process sees only GPU g
    log_path = os.path.join(WORK, f"log_{g}.txt")
    log = open(log_path, "a")
    p = subprocess.Popen([sys.executable, "-u", SCRIPT, "--work", WORK], stdout=log, stderr=subprocess.STDOUT, env=env)
    procs.append((p, log, log_path))
print(f"Started {n_procs} training process(es).")

shown = [0] * len(procs)
def show_new_lines():
    for g, (_, _, path) in enumerate(procs):
        with open(path) as f:
            lines = f.read().splitlines()
        for line in lines[shown[g]:]:
            if line.startswith("[") or "Error" in line or "Traceback" in line:
                print(f"GPU {g} {line}" if n_gpu else line)
        shown[g] = len(lines)

start = time.time()
try:
    while any(p.poll() is None for p, _, _ in procs):
        time.sleep(POLL_SECONDS)
        show_new_lines()
except KeyboardInterrupt:
    for p, _, _ in procs:
        p.terminate()                               # stopping the cell also stops the training
    print("Stopped the training processes.")
    raise
show_new_lines()
for p, log, path in procs:
    log.close()
    if p.returncode != 0:
        print(f"A training process ended with an error; see {path}")
print(f"Training finished in {(time.time() - start) / 60:.0f} minutes.")

# Part 5 — Learning Curves
Training (clean photos) and validation, measured the same way after every epoch. The dashed line is the kept epoch.

In [ ]:
from matplotlib.ticker import MaxNLocator
final_dir = os.path.join(WORK, "jobs", "final")
history = pd.read_csv(os.path.join(final_dir, "history.csv"))
with open(os.path.join(final_dir, "result.json")) as f:
    final_result = json.load(f)
best_epoch = final_result["best epoch"]
b = history[history["epoch"] == best_epoch].iloc[0]
gap = b["train_acc"] - b["val_acc"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, scale in ((ax1, "acc", 100), (ax2, "loss", 1)):
    ax.plot(history["epoch"], history[f"train_{metric}"] * scale, "o-", color="C0", label="training")
    ax.plot(history["epoch"], history[f"val_{metric}"] * scale, "s-", color="C1", label="validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1, label=f"kept epoch ({best_epoch})")
    ax.set_xlabel("Epoch"); ax.xaxis.set_major_locator(MaxNLocator(integer=True)); ax.grid(alpha=0.3); ax.legend()
ax1.set_ylabel("Accuracy (%)"); ax1.set_title("Accuracy")
ax2.set_ylabel("Loss"); ax2.set_title("Loss")
fig.suptitle(f"Final model (CLIP ViT-B/16, variant J): gap {gap * 100:+.1f} points at the kept epoch")
plt.tight_layout(); plt.show()

print(f"Kept epoch {best_epoch}: training {b['train_acc']:.1%}, validation {b['val_acc']:.1%}, "
      f"test {b['test_acc']:.1%}, gap {gap * 100:+.1f} points")
fmt = {c: "{:.1%}" for c in history.columns if c.endswith("acc")}
fmt.update({c: "{:.3f}" for c in history.columns if c.endswith("loss")})
display(history.style.format(fmt).hide(axis="index"))

# Part 6 — Results on the Test Set
The saved final model is loaded and measured on the test set from Part 1.6. The test photos were never used for training
or for choosing anything.

In [ ]:
use_amp = device.type == "cuda"
final_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
final_model.load_state_dict(torch.load(os.path.join(final_dir, "model.pt"), map_location=device))
final_model.eval()

@torch.no_grad()
def predict(net, loader):
    net.eval()
    probs, labels = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            probs.append(net(x.to(device, non_blocking=True)).float().softmax(1).cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

test_probs, test_labels = predict(final_model, test_loader)
test_probs, y_true = test_probs.numpy(), test_labels.numpy()
y_pred = test_probs.argmax(1)
print(f"Test accuracy: {(y_pred == y_true).mean():.1%} ({(y_pred == y_true).sum()} of {len(y_true)} photos)")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix: final model (test set)")
fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

pairs = sorted(((cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j), reverse=True)[:5]
print("Most common mix-ups:")
for n, i, j in pairs:
    print(f"  {class_names[i]:>20} predicted as {class_names[j]:<20} {n} times")

# Part 7 — Explainable AI
Two ways to see **why** the model predicts an action:
- **Occlusion maps:** a grey square slides over the photo; red areas are where covering the photo makes the prediction
  drop the most, so the model relies on them.
- **Attention rollout:** combines the transformer's attention across all 12 blocks, showing which parts of the photo the
  [CLS] token (the summary the classifier reads) attends to.

In [ ]:
xai_model = final_model
pred, true = y_pred, y_true
MEAN = torch.tensor(CLIP_MEAN).view(3, 1, 1)
STD = torch.tensor(CLIP_STD).view(3, 1, 1)
def to_img(t):
    return (t * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

@torch.no_grad()
def occlusion_map(x, target, size=32, stride=16):
    xs, pos = [], []
    for top in range(0, IMG_SIZE - size + 1, stride):
        for left in range(0, IMG_SIZE - size + 1, stride):
            xo = x.clone(); xo[:, top:top + size, left:left + size] = 0    # 0 = the average colour after normalization
            xs.append(xo); pos.append((top, left))
    base = xai_model(x[None].to(device)).float().softmax(1)[0, target].item()
    drops = []
    for i in range(0, len(xs), 64):
        p = xai_model(torch.stack(xs[i:i + 64]).to(device)).float().softmax(1)[:, target].cpu()
        drops.append(base - p)
    drops = torch.cat(drops)
    heat, count = torch.zeros(IMG_SIZE, IMG_SIZE), torch.zeros(IMG_SIZE, IMG_SIZE)
    for d, (top, left) in zip(drops, pos):
        heat[top:top + size, left:left + size] += d; count[top:top + size, left:left + size] += 1
    return (heat / count.clamp(min=1)).clamp(min=0).numpy()

def load_test_photo(i):
    return eval_tf(Image.open(os.path.join(TRAIN_IMG_DIR, test_df["filename"][i])).convert("RGB"))

def show_maps(rows, title):
    fig, axes = plt.subplots(len(rows), 3, figsize=(11, 3.6 * len(rows)), squeeze=False)
    for r, i in enumerate(rows):
        x = load_test_photo(i)
        p = int(pred[i]); heat = occlusion_map(x, p)
        axes[r, 0].imshow(to_img(x)); axes[r, 0].set_title(f"true: {class_names[int(true[i])]}", fontsize=10)
        axes[r, 1].imshow(to_img(x)); axes[r, 1].imshow(heat, cmap="jet", alpha=0.5)
        axes[r, 1].set_title(f"predicted: {class_names[p]} ({test_probs[i, p]:.0%})", fontsize=10)
        axes[r, 2].imshow(heat, cmap="jet"); axes[r, 2].set_title("occlusion map (red = important)", fontsize=10)
        for a in axes[r]:
            a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
correct = np.where(pred == true)[0]
wrong = np.where(pred != true)[0]
show_maps(rng.choice(correct, size=min(6, len(correct)), replace=False), "Correct predictions: where the model looks")

In [ ]:
def attention_rollout(x):
    # Combines the [CLS] attention of all transformer blocks (Abnar & Zuidema, 2020)
    store, hooks = [], []
    for blk in xai_model.blocks:
        blk.attn.fused_attn = False
        hooks.append(blk.attn.attn_drop.register_forward_hook(lambda m, i, o: store.append(i[0].float().mean(1)[0].cpu())))
    with torch.no_grad():
        xai_model(x[None].to(device))
    for h in hooks:
        h.remove()
    roll = torch.eye(store[0].shape[-1])
    for a in store:
        a = a + torch.eye(a.shape[-1]); a = a / a.sum(-1, keepdim=True)
        roll = a @ roll
    n = int(round((roll.shape[-1] - 1) ** 0.5))
    m = roll[0, 1:1 + n * n].reshape(1, 1, n, n)
    return F.interpolate(m, size=(IMG_SIZE, IMG_SIZE), mode="bilinear")[0, 0].numpy()

sample = rng.choice(correct, size=min(6, len(correct)), replace=False)
fig, axes = plt.subplots(2, len(sample), figsize=(3.2 * len(sample), 6.6), squeeze=False)
for c, i in enumerate(sample):
    x = load_test_photo(i)
    axes[0, c].imshow(to_img(x)); axes[0, c].set_title(class_names[int(pred[i])], fontsize=10)
    axes[1, c].imshow(to_img(x)); axes[1, c].imshow(attention_rollout(x), cmap="jet", alpha=0.5)
    axes[0, c].axis("off"); axes[1, c].axis("off")
fig.suptitle("Attention rollout: what the transformer attends to (bottom row)"); plt.tight_layout(); plt.show()

examples = [k for _, i, j in pairs[:3] for k in wrong if true[k] == i and pred[k] == j][:6]
if examples:
    show_maps(examples, "Mistakes: what the model looked at when it got it wrong")
conf = test_probs.max(1)
plt.figure(figsize=(7, 3.5))
plt.hist(conf[pred == true], bins=20, range=(0, 1), alpha=0.7, label="correct")
plt.hist(conf[pred != true], bins=20, range=(0, 1), alpha=0.7, label="wrong")
plt.xlabel("Confidence of the prediction"); plt.ylabel("Test photos"); plt.legend()
plt.title("Wrong predictions are usually less confident"); plt.tight_layout(); plt.show()

# Part 8 — Predictions on New Photos
The dataset comes with **5,400 extra photos without labels** (the `test` folder, from the original competition, whose answers
were kept secret). The model predicts an action for each one; the predictions are saved to `test_predictions.csv`,
and 12 random photos are shown with the prediction underneath. There's no answer key, so judge by eye.

In [ ]:
kaggle_test_df = pd.read_csv(os.path.join(DATA_DIR, "Testing_set.csv"))
kaggle_test_loader = make_loader(kaggle_test_df, TEST_IMG_DIR, eval_tf, train=False, labeled=False)
kaggle_probs, _ = predict(final_model, kaggle_test_loader)
kaggle_test_df["label"] = [class_names[i] for i in kaggle_probs.argmax(dim=1).numpy()]
kaggle_test_df["confidence"] = kaggle_probs.max(dim=1).values.numpy().round(3)
kaggle_test_df.to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"), index=False)
print("Saved predictions for all", len(kaggle_test_df), "photos to", os.path.join(OUTPUT_DIR, "test_predictions.csv"))

# A plain-English description for each action
DESCRIPTIONS = {
    "calling": "Talking on the phone", "clapping": "Clapping", "cycling": "Riding a bicycle",
    "dancing": "Dancing", "drinking": "Drinking", "eating": "Eating", "fighting": "Fighting",
    "hugging": "Hugging", "laughing": "Laughing", "listening_to_music": "Listening to music",
    "running": "Running", "sitting": "Sitting", "sleeping": "Sleeping",
    "texting": "Texting on a phone", "using_laptop": "Using a laptop",
}

def show_prediction(ax, img, probs):
    # Draws the photo with the prediction written underneath it
    best, second = probs.argsort(descending=True)[:2].tolist()
    ax.imshow(img)
    ax.axis("off")
    lines = [(DESCRIPTIONS[class_names[best]], "bold", 13),
             (f"{class_names[best]} · {probs[best]:.0%} confident", "normal", 11)]
    if probs[best] < 0.70:   # not sure: also show the second guess
        lines.append((f"2nd guess: {class_names[second]} ({probs[second]:.0%})", "normal", 10))
    for k, (text, weight, size) in enumerate(lines):
        ax.annotate(text, xy=(0.5, 0), xycoords="axes fraction", xytext=(0, -8 - 18 * k),
                    textcoords="offset points", ha="center", va="top", fontsize=size, fontweight=weight)

demo = kaggle_test_df.sample(12, random_state=7)
fig, axes = plt.subplots(3, 4, figsize=(16, 15))
for ax, (i, row) in zip(axes.flat, demo.iterrows()):
    show_prediction(ax, Image.open(os.path.join(TEST_IMG_DIR, row["filename"])).convert("RGB"), kaggle_probs[i])
plt.subplots_adjust(hspace=0.5, wspace=0.1)
plt.show()

## 8.1 Try the model on any photo
`predict_photo` shows one photo with the model's prediction underneath. It works with a file path or a web address:
right-click a photo on a website, choose **Copy image address**, and paste it between the quotes
(Kaggle's **Internet** setting must be on). The model only knows these 15 actions, so it always picks one of them;
a low confidence and a second guess mean it isn't sure.

In [ ]:
import io, urllib.request

def predict_photo(source):
    # source: a web address of a photo (https://...) or a file path
    if str(source).startswith("http"):
        request = urllib.request.Request(source, headers={"User-Agent": "Mozilla/5.0"})
        img = Image.open(io.BytesIO(urllib.request.urlopen(request, timeout=20).read())).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    final_model.eval()
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        probs = final_model(x).float().softmax(dim=1)
    fig, ax = plt.subplots(figsize=(6, 6))
    show_prediction(ax, img, probs[0].cpu())
    plt.show()

predict_photo(os.path.join(TEST_IMG_DIR, "Image_1.jpg"))

# Part 9 — 5-Fold Cross-Validation
One split can be lucky or unlucky: the test accuracy in Part 6 depends partly on *which* photos ended up in the test set.
**Cross-validation** repeats the whole training on 5 different splits (Part 3.2): every photo is in a test fold exactly once,
and no model is tested on photos it trained on or used for early stopping.
- The **mean** test accuracy is the best estimate of the accuracy on new photos.
- The **standard deviation** shows how much the result depends on the split. Around one point or less means it is stable.
- The curves of the five folds show whether the good fit (small gap) is repeatable, not a lucky run.

In [ ]:
cv_rows, cv_hist = [], {}
for job in jobs[1:]:
    folder = os.path.join(WORK, "jobs", job["name"])
    if not os.path.exists(os.path.join(folder, "result.json")):
        print(f"{job['name']}: no result (see the logs in Part 4)")
        continue
    with open(os.path.join(folder, "result.json")) as f:
        cv_rows.append(json.load(f))
    cv_hist[job["name"]] = pd.read_csv(os.path.join(folder, "history.csv"))
assert cv_rows, "No fold finished. Read the error lines printed in Part 4."
cv_df = pd.DataFrame(cv_rows).set_index("training")
mean_acc, std_acc = cv_df["test acc"].mean(), cv_df["test acc"].std(ddof=1)
mean_gap, std_gap = cv_df["train-val gap"].mean(), cv_df["train-val gap"].std(ddof=1)
print(f"{len(cv_df)}-fold cross-validation test accuracy: {mean_acc:.1%} ± {std_acc:.1%}  (mean ± standard deviation)")
print(f"Lowest fold: {cv_df['test acc'].min():.1%}   Highest fold: {cv_df['test acc'].max():.1%}")
print(f"Train-validation gap: {mean_gap * 100:+.1f} ± {std_gap * 100:.1f} points")
print(f"Final model on its own test set (Part 6): {(y_pred == y_true).mean():.1%}")
cv_df.to_csv(os.path.join(OUTPUT_DIR, "cross_validation.csv"))
display(cv_df.style.format({"train acc": "{:.1%}", "val acc": "{:.1%}", "test acc": "{:.1%}",
                            "train-val gap": "{:+.1%}", "minutes": "{:.0f}"}))

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar([n.replace("fold_", "") for n in cv_df.index], cv_df["test acc"] * 100, color="#4c78a8")
ax.bar_label(bars, labels=[f"{a:.1%}" for a in cv_df["test acc"]], padding=3)
ax.axhline(mean_acc * 100, color="black", linestyle="--", linewidth=1, zorder=4, label=f"mean {mean_acc:.1%}")
ax.axhspan((mean_acc - std_acc) * 100, (mean_acc + std_acc) * 100, color="gray", alpha=0.35, zorder=3,
           label=f"mean ± 1 standard deviation ({std_acc:.1%})")
ax.set_ylim(0, 100)
ax.set_xlabel("Fold (the part of the data used for testing)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title(f"{len(cv_df)}-fold cross-validation: accuracy on each held-out fold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(cv_hist), figsize=(4.2 * len(cv_hist), 3.8), sharey=True, squeeze=False)
for ax, (name, h) in zip(axes[0], cv_hist.items()):
    r = cv_df.loc[name]
    ax.plot(h["epoch"], h["train_acc"] * 100, "o-", color="C0", ms=3, label="training")
    ax.plot(h["epoch"], h["val_acc"] * 100, "s-", color="C1", ms=3, label="validation")
    ax.axvline(r["best epoch"], color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"{name.replace('_', ' ')}: gap {r['train-val gap'] * 100:+.1f} pts", fontsize=10)
    ax.set_xlabel("Epoch"); ax.xaxis.set_major_locator(MaxNLocator(integer=True)); ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("Accuracy (%)"); axes[0, 0].legend(fontsize=8)
fig.suptitle("Learning curves of the five cross-validation folds")
plt.tight_layout(); plt.show()

# Summary

In [ ]:
print("Final model: CLIP ViT-B/16 image encoder, fine-tuned (variant J from notebook 07)")
print(f"  Settings: {FREEZE_BLOCKS} of 12 blocks frozen, learning rate {LEARNING_RATE}, dropout {DROPOUT}, "
      f"drop path {DROP_PATH}, weight decay {WEIGHT_DECAY}, strong augmentation, mixup / cutmix")
print(f"  Training: {len(history)} epochs (early stopping), kept epoch {best_epoch}")
print(f"  Accuracy at the kept epoch: training {b['train_acc']:.1%}, validation {b['val_acc']:.1%}, "
      f"gap {gap * 100:+.1f} points")
print(f"  Test accuracy: {(y_pred == y_true).mean():.1%} on {len(y_true)} photos never used for training or choosing")
print(f"  5-fold cross-validation: {mean_acc:.1%} ± {std_acc:.1%} test accuracy, "
      f"gap {mean_gap * 100:+.1f} ± {std_gap * 100:.1f} points")